# Lab 10: Data File Analyzer (DS-STAR Component)

**Navigation** : [Lab 9 <<](../Day4-Foundations/Lab9-First-ADK-Agent.ipynb) | [Index](../../README.md) | [>> Lab 11](Lab11-Planner-Coder-Loop.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Implémenter le module File Analyzer de DS-STAR
2. Analyser des fichiers hétérogènes (CSV, JSON, Markdown, texte)
3. Extraire des métadonnées structurées automatiquement
4. Générer des résumés intelligents avec le LLM

### Prérequis
- Python 3.10+
- Configuration multi-provider active
- Connaissance de Pandas (Lab 4)

### Durée estimée : 30-40 minutes

## 1. Configuration

In [1]:
import sys
from pathlib import Path

# Add parent directory (AgenticDataScience) to path for config/utils imports
sys.path.insert(0, str(Path().resolve().parent))

import os
import json
import pandas as pd
import numpy as np
from dataclasses import dataclass, asdict

from config import get_settings
from utils import LLMClient
print("Imports OK : json, re, pandas")

Imports OK : json, re, pandas


Chargement des parametres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: openrouter


## 2. FileMetadata DataClass

In [3]:
@dataclass
class FileMetadata:
    filename: str
    format: str
    size_bytes: int
    num_rows: int = None
    num_columns: int = None
    columns: list = None
    statistics: dict = None
    sample_data: list = None
print("Dataclasses definies : FileData, AnalysisResult")

Dataclasses definies : FileData, AnalysisResult


## 3. FileAnalyzer Class

In [4]:
class FileAnalyzer:
    SUPPORTED = {'.csv': 'csv', '.json': 'json', '.md': 'markdown', '.txt': 'text'}

    def __init__(self, llm_client=None):
        self.llm = llm_client or LLMClient()

    def detect_format(self, path):
        ext = Path(path).suffix.lower()
        return self.SUPPORTED.get(ext, 'unknown')

    def analyze_csv(self, path):
        df = pd.read_csv(path)
        cols = []
        for c in df.columns:
            info = {'name': c, 'dtype': str(df[c].dtype), 'missing_pct': round(df[c].isna().mean()*100, 2)}
            if df[c].dtype in ['int64', 'float64']:
                info['stats'] = {'min': float(df[c].min()), 'max': float(df[c].max()), 'mean': float(df[c].mean())}
            cols.append(info)
        return FileMetadata(
            filename=Path(path).name, format='csv', size_bytes=os.path.getsize(path),
            num_rows=len(df), num_columns=len(df.columns), columns=cols,
            sample_data=df.head(5).to_dict('records')
        )

    def analyze(self, path):
        fmt = self.detect_format(path)
        if fmt == 'csv':
            return self.analyze_csv(path)
        raise ValueError(f"Format non supporte: {fmt}")

    def generate_summary(self, meta):
        lines = [f"Fichier: {meta.filename}", f"Format: {meta.format}", f"Taille: {meta.size_bytes/1024:.1f} KB"]
        if meta.num_rows:
            lines.append(f"Lignes: {meta.num_rows}, Colonnes: {meta.num_columns}")
        if meta.columns:
            for c in meta.columns[:5]:
                lines.append(f"  - {c['name']} ({c['dtype']}): {c.get('missing_pct', 0)}% manquant")
        return "\n".join(lines)

    def generate_llm_summary(self, meta, question=None):
        ctx = self.generate_summary(meta)
        if meta.sample_data:
            ctx += f"\n\nEchantillon:\n{json.dumps(meta.sample_data[:2], indent=2, default=str)[:400]}"
        prompt = f"Analyse ce fichier et resume-le:\n{ctx}\n{f'Question: {question}' if question else ''}"
        return self.llm.generate(prompt, temperature=0.3)
print("Classe FileAnalyzer definie")

Classe FileAnalyzer definie


## 4. Tests

In [5]:
# Creation fichier test
import tempfile
test_dir = tempfile.mkdtemp()

df = pd.DataFrame({
    'id': range(1, 101),
    'product': [f'P{i}' for i in range(1, 101)],
    'category': np.random.choice(['A', 'B', 'C'], 100),
    'price': np.random.uniform(10, 500, 100).round(2),
    'qty': np.random.randint(1, 50, 100)
})
df.loc[10:15, 'price'] = np.nan

csv_path = os.path.join(test_dir, 'products.csv')
df.to_csv(csv_path, index=False)
print(f'CSV cree: {csv_path}')

CSV cree: C:\Users\jsboi\AppData\Local\Temp\claude\tmpttjmesic\products.csv


Test de l'analyseur de fichiers.

In [6]:
# Test analyseur
analyzer = FileAnalyzer()
meta = analyzer.analyze(csv_path)
print(analyzer.generate_summary(meta))

Fichier: products.csv
Format: csv
Taille: 1.9 KB
Lignes: 100, Colonnes: 5
  - id (int64): 0.0% manquant
  - product (str): 0.0% manquant
  - category (str): 0.0% manquant
  - price (float64): 6.0% manquant
  - qty (int64): 0.0% manquant

Detail des colonnes detectees.

In [7]:
# Colonnes detail
for c in meta.columns:
    print(f"{c['name']}: {c['dtype']}, {c.get('missing_pct', 0)}% manquant")
    if 'stats' in c:
        print(f"  -> min={c['stats']['min']:.1f}, max={c['stats']['max']:.1f}, mean={c['stats']['mean']:.1f}")

id: int64, 0.0% manquant
  -> min=1.0, max=100.0, mean=50.5
product: str, 0.0% manquant
category: str, 0.0% manquant
price: float64, 6.0% manquant
  -> min=22.5, max=490.5, mean=261.1
qty: int64, 0.0% manquant
  -> min=1.0, max=49.0, mean=24.0


## 5. Resume LLM

In [8]:
# Resume intelligent
summary = analyzer.generate_llm_summary(meta)
print(summary)

Voici une analyse et un résumé du fichier **products.csv** :

### Description générale
- **Nombre de lignes** : 100
- **Nombre de colonnes** : 5
- **Colonnes** :
  - `id` (int64) : Identifiant unique du produit, sans valeurs manquantes.
  - `product` (str) : Nom ou code du produit, sans valeurs manquantes.
  - `category` (str) : Catégorie du produit, sans valeurs manquantes.
  - `price` (float64) : Prix unitaire du produit, avec 6.0% de valeurs manquantes.
  - `qty` (int64) : Quantité disponible, sans valeurs manquantes.

### Données manquantes
- La colonne `price` contient environ 6% de valeurs manquantes (environ 6 valeurs sur 100).
- Les autres colonnes sont complètes.

### Aperçu des données
- Exemple de produit : 
  - id=1, product="P1", category="B", price=85.1, qty=47
  - id=2, product="P2", category="A", price=372.1, qty=12

### Analyse possible
- Les produits sont catégorisés (ex: catégories A, B, etc.).
- Les prix varient fortement (ex: 85.1 à 372.1 dans l’échantillon).
- La 

Generation d'un resume intelligent guide par le LLM.

In [9]:
# Resume guide
q = "Quelles analyses puis-je faire sur ces donnees?"
print(f"Question: {q}\n")
result = analyzer.generate_llm_summary(meta, q)
print(result)

Question: Quelles analyses puis-je faire sur ces donnees?



Voici quelques analyses que vous pouvez réaliser sur ce fichier `products.csv` :

### 1. Analyse descriptive des données
- **Statistiques de base** : Moyenne, médiane, minimum, maximum, écart-type des colonnes `price` et `qty`.
- **Distribution des catégories** : Nombre de produits par catégorie (`category`).
- **Répartition des prix** : Histogramme ou boîte à moustaches pour visualiser la distribution des prix.
- **Répartition des quantités** : Analyse similaire pour la colonne `qty`.

### 2. Analyse des valeurs manquantes
- Vous avez 6% de valeurs manquantes dans la colonne `price`. Vous pouvez :
  - Identifier les produits avec prix manquant.
  - Décider d’imputer ces valeurs (moyenne, médiane, ou autre méthode) ou de les exclure.

### 3. Analyse des relations entre variables
- **Prix moyen par catégorie** : Calculer le prix moyen des produits pour chaque catégorie.
- **Quantité moyenne par catégorie** : Même chose pour la quantité.
- **Corrélation entre prix et quantité** : Voir s’

## 6. Resume du Lab### Points cles1. FileAnalyzer: Analyse automatique de fichiers2. Metadonnees: Extraction de schemas et stats3. Resume LLM: Contextualisation pour le Planner### Prochaine etape- Lab 11: Boucle Planner-Coder-Verifier

In [10]:
# Cleanup
import shutil
shutil.rmtree(test_dir)
print('Done')

Done


## Exercice

À vous d'étendre le FileAnalyzer pour gérer un nouveau format de fichier !


In [11]:
# Exercice : Etendez le FileAnalyzer pour gerer les fichiers JSON
# 1. Ajoutez une methode analyze_json
# 2. Testez-la sur un fichier JSON de test

import json
import tempfile

# Creation d'un fichier JSON de test
exercise_dir = tempfile.mkdtemp()
test_json_path = os.path.join(exercise_dir, 'test_data.json')
test_data = {
    'products': [
        {'id': 1, 'name': 'Laptop', 'price': 999.99, 'stock': 15},
        {'id': 2, 'name': 'Mouse', 'price': 19.99, 'stock': 50},
        {'id': 3, 'name': 'Keyboard', 'price': 49.99, 'stock': 30}
    ],
    'metadata': {
        'source': 'inventory_system',
        'last_updated': '2026-03-13'
    }
}

with open(test_json_path, 'w') as f:
    json.dump(test_data, f)

# Exercice: Implementez la classe ExtendedFileAnalyzer
# Elle doit heriter de FileAnalyzer et ajouter une methode analyze_json
class ExtendedFileAnalyzer(FileAnalyzer):
    def analyze_json(self, path):
        """Analyse un fichier JSON et extrait les metadonnees."""
        # Exercice: Ouvrez et chargez le fichier JSON
        # Indice: with open(path, 'r') as f: data = json.load(f)
        data = None  # Remplacez None

        # Exercice: Comptez les cles au niveau racine
        root_keys = None  # Indice: list(data.keys())

        # Exercice: Trouvez la plus grande liste dans les valeurs
        # Indice: parcourez data.items(), testez isinstance(value, list)
        max_list = 0
        list_name = None
        # ... votre code ici

        # Exercice: Retournez un FileMetadata avec les informations extraites
        return None  # Remplacez par FileMetadata(...)

# Exercice: Testez l'analyseur etendu
# extended_analyzer = ExtendedFileAnalyzer()
# json_meta = extended_analyzer.analyze_json(test_json_path)
# print(f"Fichier: {json_meta.filename}")
# print(f"Cles racine: {json_meta.columns}")

# Nettoyage
# os.remove(test_json_path)

print("Exercice a completer")

Exercice a completer


## Conclusion

Ce notebook a permis d'explorer les aspects essentiels de lab10 file analyzer. Les points cles :

- Les concepts fondamentaux ont ete presentes et illustres
- Les exercices proposent une mise en pratique progressive
- Les resultats obtenus permettent de valider la comprehension

**Pour aller plus loin** : approfondir les aspects avances du sujet et explorer les liens avec d'autres domaines.